# なす袋デザインの検討　AI画像生成

デザインA（高級・ブランド感）の写真をAI（Google の Gemini 画像モデル）に渡して、いろいろな案の写真を作り、見比べて考えるためのノートブックです。

**最初に1回だけ**
1. https://aistudio.google.com で APIキーを作り、課金（Billing）を有効にする
2. 左の **鍵マーク** →「新しいシークレットを追加」→ 名前 `GOOGLE_API_KEY`、値に APIキー →「ノートブックからのアクセス」をオン

**使い方**：上から順に ▶ を押します。案を変えたいときは ③ の指示文を書き換えて、③と④をもう一度 ▶。

In [ ]:
#@title ① 準備
!pip -q install -U google-genai pillow
import io, os, re, time, datetime
from PIL import Image
from IPython.display import display
from google import genai
from google.genai import types
from google.colab import userdata, files

client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

# 使える画像モデルを問い合わせて、いちばん新しい上位モデルを選ぶ（名前は時期で変わるため決め打ちしない）
names = sorted({m.name.split('/')[-1] for m in client.models.list()
                if m.name.split('/')[-1].startswith('gemini') and 'image' in m.name})
def score(n):
    v = re.search(r'gemini-(\d+(?:\.\d+)?)', n)
    return ('pro' in n, float(v.group(1)) if v else 0, 'preview' not in n)
MODEL = max(names, key=score) if names else None
print('使える画像モデル:', names)
print('使うモデル:', MODEL, '（変えたいときは次の行のように書く： MODEL = "gemini-2.5-flash-image"）')


In [ ]:
#@title ② 見本写真をアップロード
up = files.upload()
REF_NAME, REF_BYTES = next(iter(up.items()))
REF_MIME = 'image/png' if REF_NAME.lower().endswith('.png') else 'image/jpeg'
ref = Image.open(io.BytesIO(REF_BYTES)); print(REF_NAME, ref.size)
display(ref.resize((ref.width * 400 // ref.height, 400)))


In [ ]:
#@title ③ 指示文（ここを書き換えて案を考える）
#@markdown 1案につき何枚作るか
N = 2  #@param {type:"slider", min:1, max:6, step:1}
#@markdown 縦横比
ASPECT = "3:4"  #@param ["3:4", "4:5", "2:3", "1:1", "9:16"]

# 全部の案に共通する前置き
BASE = """添付の写真は、高知県安芸市のなすを入れる透明な防曇袋（平袋）のデザイン見本です。
写真の雰囲気（透明な袋のつやとしわ、白インクの印刷、金色の筆記体、背景の緑のぼけ、木のテーブル、写真のリアルさ）は保ち、
袋の印刷を次のとおり作り直した写真を1枚作ってください。日本語の文字は一字一句正確に、くっきりと。
写真の上の紺色の見出し帯（A 高級・ブランド感）は入れず、袋の写真だけにしてください。
"""

# 案（名前: 指示）。好きなだけ足したり消したりしてよい
IDEAS = {
 "1_ロゴを消して団体名": """- 太陽と山のロゴマーク、「せんとうふぁーむ」「SENTO FARM」は消す
- その位置に小さく「安芸なす生産者グループ」、その下に大きく「ベジワン」、その下に英字「VEGE ONE」
- 「こだわりの高知なす」を「こだわりの安芸ナス」に変える。ほかはそのまま""",
 "2_M5本入_紫の枠": """- 1の内容に加えて、なすを5本（やや細めのMサイズ）にする
- 袋の印刷に、なす全体を囲む太めの紫の枠線を入れ、枠の上の中央に紫の札で白抜き「M・5本入」""",
 "3_L3本入_緑の枠": """- 1の内容に加えて、なすは3本（太めのLサイズ）
- 袋の印刷に、なす全体を囲む太めの緑の枠線を入れ、枠の上の中央に緑の札で白抜き「L・3本入」""",
 "4_上に色の帯": """- 1の内容に加えて、袋の上部に横いっぱいの紫の帯を入れ、白抜きで「M・5本入」。なすは5本""",
 "5_もっと高級に": """- 1の内容に加えて、白とゴールドの配分を見直し、余白を広くとって、百貨店のギフト売り場に並ぶような上質さにする
- 文字は少なく、ロゴは入れない""",
}
print(len(IDEAS), '案 ×', N, '枚 =', len(IDEAS) * N, '枚 作ります')


In [ ]:
#@title ④ 画像を作る・並べる・保存
OUT = '/content/なす袋AI画像/' + datetime.datetime.now().strftime('%Y%m%d_%H%M')
os.makedirs(OUT, exist_ok=True)

SIZE = {'image_size': '2K'}
def make(prompt):
    for attempt in range(3):
        try:
            cfg = types.GenerateContentConfig(response_modalities=['IMAGE', 'TEXT'],
                                              image_config=types.ImageConfig(aspect_ratio=ASPECT, **SIZE))
            r = client.models.generate_content(model=MODEL, config=cfg,
                    contents=[types.Part.from_bytes(data=REF_BYTES, mime_type=REF_MIME), prompt])
            texts = []
            for c in (r.candidates or []):
                for p in (c.content.parts if c.content and c.content.parts else []):
                    if p.inline_data is not None and p.inline_data.data:
                        return Image.open(io.BytesIO(p.inline_data.data)).convert('RGB'), ''
                    if p.text: texts.append(p.text)
            return None, ' / '.join(texts) or '画像が返ってきませんでした'
        except Exception as e:
            m = str(e)
            if 'image_size' in m or 'imageSize' in m: SIZE.clear(); continue     # 画質指定に未対応のモデル
            if any(k in m for k in ('429', 'RESOURCE_EXHAUSTED', '503', 'UNAVAILABLE')): time.sleep(10 * (attempt + 1)); continue
            return None, m
    return None, 'やり直しても作れませんでした'

for name, idea in IDEAS.items():
    imgs = []
    for i in range(N):
        img, why = make(BASE + idea)
        if img is None: print('×', name, why[:200]); continue
        img.save(f'{OUT}/{name}_{i+1}.jpg', quality=95); imgs.append(img)
    if imgs:
        print('■', name)
        h = 420; th = [im.resize((im.width * h // im.height, h)) for im in imgs]
        row = Image.new('RGB', (sum(t.width for t in th) + 10 * (len(th) + 1), h + 20), (26, 33, 69))
        x = 10
        for t in th: row.paste(t, (x, 10)); x += t.width + 10
        display(row)
print('保存先:', OUT, '（左のフォルダのアイコンから右クリックでダウンロードできます）')


## 気に入った写真を Googleドライブに残す（任意）
下のセルを ▶ すると、作った写真をすべて「マイドライブ / なす袋AI画像」にコピーします。

In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')
dst = '/content/drive/MyDrive/なす袋AI画像/' + os.path.basename(OUT)
shutil.copytree(OUT, dst, dirs_exist_ok=True); print('コピーしました:', dst)
